MLA toy example. The file is stored at "e:\Ridwanur\Documents\Y3 FYM\toy_mla.py"
Use command: $env:PYTHONIOENCODING='utf-8'; python "e:\Ridwanur\Documents\Y3 FYM\toy_mla.py"

======================================================================
  MLA (Multi-head Latent Attention) — Toy Simulation
======================================================================

  d_model      = 4
  n_heads      = 2
  head_dim     = 2   (content dimension per head)
  rope_dim     = 2   (decoupled positional dimension per head)
  kv_lora_rank = 2   (latent bottleneck dimension)

  Attention dim per head = head_dim + rope_dim = 4
    (content and position are CONCATENATED for score computation)

  === Cache Size Comparison (per token) ===
  Standard MHA : n_heads x head_dim x 2 (K+V)     = 8 values
  GQA (kv=1)   : n_kv_heads x head_dim x 2 (K+V)  = 4 values
  MLA          : kv_lora_rank + rope_dim            = 4 values
                 (c_kv latent)   (positional key)

  In real DeepSeek-V3 (n_heads=128, head_dim=128, kv_lora_rank=512, rope_dim=64):
    MHA cache/token: 128 x 128 x 2 = 32,768 values
    MLA cache/token: 512 + 64      =    576 values  --> 57x smaller!


######################################################################
  TRAINING MODE  (full sequence, no cache)
######################################################################

  -- Input X  [5, 4] --
    A    [1.0, 0.0, 2.0, 1.0]
    B    [0.0, 1.0, 1.0, 0.0]
    C    [2.0, 1.0, 0.0, 1.0]
    D    [1.0, 2.0, 1.0, 0.0]
    E    [0.0, 0.0, 2.0, 1.0]

======================================================================
  QUERY PATH:  two separate projections (content + position)
======================================================================

  -- q_content_flat  (X @ W_q_content)  [5, 4] --
    A    [3.0, 2.0, 1.0, 2.0]
    B    [1.0, 2.0, 1.0, 0.0]
    C    [2.0, 1.0, 2.0, 3.0]
    D    [2.0, 3.0, 2.0, 1.0]
    E    [2.0, 2.0, 1.0, 1.0]

  Reshape to heads: [1, 5, 4] -> [1, 2, 5, 2]

  -- q_content Head 0  [5, 2] --
    A    [3.0, 2.0]
    B    [1.0, 2.0]
    C    [2.0, 1.0]
    D    [2.0, 3.0]
    E    [2.0, 2.0]

  -- q_content Head 1  [5, 2] --
    A    [1.0, 2.0]
    B    [1.0, 0.0]
    C    [2.0, 3.0]
    D    [2.0, 1.0]
    E    [1.0, 1.0]

  q_rope: X @ W_q_rope -> reshape -> apply RoPE

  -- q_rope (with position info) Head 0  [5, 2] --
    A    [2.0, 1.0]
    B    [0.1, 1.1]
    C    [3.2, 2.2]
    D    [1.3, 2.3]
    E    [1.4, 1.4]

  -- q_rope (with position info) Head 1  [5, 2] --
    A    [3.0, 2.0]
    B    [1.1, 2.1]
    C    [2.2, 1.2]
    D    [2.3, 3.3]
    E    [2.4, 2.4]

======================================================================
  KV PATH:  compress -> (later) decompress
======================================================================

  COMPRESSION:  X @ W_kv_down
    [1, 5, 4] @ [4, 2] -> [1, 5, 2]
    4 dims compressed to 2 dims per token!

  -- c_kv (compressed latent)  [5, 2] --
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]
    D    [2.0, 3.0]
    E    [2.0, 3.0]

  k_rope: X @ W_k_rope -> reshape to (B, 1, S, rope_dim) -> apply RoPE
    Shape: [1, 1, 5, 2]  <-- 1 head, shared across all 2 query heads

  -- k_rope (shared, with position) Head 0  [5, 2] --
    A    [2.0, 3.0]
    B    [2.1, 1.1]
    C    [1.2, 2.2]
    D    [3.3, 2.3]
    E    [2.4, 2.4]

  DECOMPRESSION:  c_kv @ W_kv_up
    [1, 5, 2] @ [2, 8] -> [1, 5, 8]

  -- kv_content (decompressed)  [5, 8] --
    A    [3.0, 3.0, 3.0, 6.0, 3.0, 3.0, 3.0, 3.0]
    B    [1.0, 2.0, 1.0, 3.0, 2.0, 1.0, 2.0, 1.0]
    C    [2.0, 2.0, 2.0, 4.0, 2.0, 2.0, 2.0, 2.0]
    D    [2.0, 3.0, 2.0, 5.0, 3.0, 2.0, 3.0, 2.0]
    E    [2.0, 3.0, 2.0, 5.0, 3.0, 2.0, 3.0, 2.0]

  Reshape to (B, n_heads, S, 2*head_dim) then chunk into K_content + V:
    k_content: [1, 2, 5, 2]
    v:         [1, 2, 5, 2]

  -- k_content Head 0  [5, 2] --
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]
    D    [2.0, 3.0]
    E    [2.0, 3.0]

  -- k_content Head 1  [5, 2] --
    A    [3.0, 3.0]
    B    [2.0, 1.0]
    C    [2.0, 2.0]
    D    [3.0, 2.0]
    E    [3.0, 2.0]

  -- v (values) Head 0  [5, 2] --
    A    [3.0, 6.0]
    B    [1.0, 3.0]
    C    [2.0, 4.0]
    D    [2.0, 5.0]
    E    [2.0, 5.0]

  -- v (values) Head 1  [5, 2] --
    A    [3.0, 3.0]
    B    [2.0, 1.0]
    C    [2.0, 2.0]
    D    [3.0, 2.0]
    E    [3.0, 2.0]

======================================================================
  ASSEMBLE:  concatenate content + position dims
======================================================================

  k_rope expanded: [1, 1, 5, 2] -> [1, 2, 5, 2]
    (broadcast the shared positional key to all 2 heads)

  q_full = cat(q_content, q_rope) along last dim:
    [1, 2, 5, 2] + [1, 2, 5, 2] -> [1, 2, 5, 4]
    Each head: [2 content dims | 2 position dims] = 4 total

  -- q_full  [content | position] Head 0  [5, 4] --
    A    [3.0, 2.0, 2.0, 1.0]
    B    [1.0, 2.0, 0.1, 1.1]
    C    [2.0, 1.0, 3.2, 2.2]
    D    [2.0, 3.0, 1.3, 2.3]
    E    [2.0, 2.0, 1.4, 1.4]

  -- q_full  [content | position] Head 1  [5, 4] --
    A    [1.0, 2.0, 3.0, 2.0]
    B    [1.0, 0.0, 1.1, 2.1]
    C    [2.0, 3.0, 2.2, 1.2]
    D    [2.0, 1.0, 2.3, 3.3]
    E    [1.0, 1.0, 2.4, 2.4]

  k_full = cat(k_content, k_rope_expanded) along last dim:

  -- k_full  [content | position] Head 0  [5, 4] --
    A    [3.0, 3.0, 2.0, 3.0]
    B    [1.0, 2.0, 2.1, 1.1]
    C    [2.0, 2.0, 1.2, 2.2]
    D    [2.0, 3.0, 3.3, 2.3]
    E    [2.0, 3.0, 2.4, 2.4]

  -- k_full  [content | position] Head 1  [5, 4] --
    A    [3.0, 3.0, 2.0, 3.0]
    B    [2.0, 1.0, 2.1, 1.1]
    C    [2.0, 2.0, 1.2, 2.2]
    D    [3.0, 2.0, 3.3, 2.3]
    E    [3.0, 2.0, 2.4, 2.4]

======================================================================
  ATTENTION:  q_full @ k_full^T / sqrt(4)
======================================================================

  Causal Mask:
    query v  key ->  ['A', 'B', 'C', 'D', 'E']
    A                ok -in -in -in -in
    B                ok  ok -in -in -in
    C                ok  ok  ok -in -in
    D                ok  ok  ok  ok -in
    E                ok  ok  ok  ok  ok

  -- Head 0 Masked Scores --
    query v  key ->  ['A', 'B', 'C', 'D', 'E']
    A              [' 11.00', '  -inf', '  -inf', '  -inf', '  -inf']
    B              ['  6.25', '  3.21', '  -inf', '  -inf', '  -inf']
    C              [' 11.00', '  6.57', '  7.34', '  -inf', '  -inf']
    D              [' 12.25', '  6.63', '  8.31', ' 11.29', '  -inf']
    E              ['  9.50', '  5.24', '  6.38', '  8.92', '  8.36']

  -- Head 1 Masked Scores --
    query v  key ->  ['A', 'B', 'C', 'D', 'E']
    A              [' 10.50', '  -inf', '  -inf', '  -inf', '  -inf']
    B              ['  5.75', '  3.31', '  -inf', '  -inf', '  -inf']
    C              [' 11.50', '  6.47', '  7.64', '  -inf', '  -inf']
    D              [' 11.75', '  6.73', '  8.01', ' 11.59', '  -inf']
    E              ['  9.00', '  5.34', '  6.08', '  9.22', '  8.26']

  -- Head 0 Attention Weights (softmax) --
    query v  key ->  ['A', 'B', 'C', 'D', 'E']
    A              ['  1.00', '  0.00', '  0.00', '  0.00', '  0.00']
    B              ['  0.95', '  0.05', '  0.00', '  0.00', '  0.00']
    C              ['  0.96', '  0.01', '  0.02', '  0.00', '  0.00']
    D              ['  0.71', '  0.00', '  0.01', '  0.27', '  0.00']
    E              ['  0.52', '  0.01', '  0.02', '  0.29', '  0.17']

  -- Head 1 Attention Weights (softmax) --
    query v  key ->  ['A', 'B', 'C', 'D', 'E']
    A              ['  1.00', '  0.00', '  0.00', '  0.00', '  0.00']
    B              ['  0.92', '  0.08', '  0.00', '  0.00', '  0.00']
    C              ['  0.97', '  0.01', '  0.02', '  0.00', '  0.00']
    D              ['  0.53', '  0.00', '  0.01', '  0.45', '  0.00']
    E              ['  0.36', '  0.01', '  0.02', '  0.44', '  0.17']

  Output = weights @ V  (note: V is head_dim=2, not 4)

  -- Attention output Head 0  [5, 2] --
    A    [3.0, 6.0]
    B    [2.91, 5.86]
    C    [2.95, 5.92]
    D    [2.71, 5.69]
    E    [2.51, 5.48]

  -- Attention output Head 1  [5, 2] --
    A    [3.0, 3.0]
    B    [2.92, 2.84]
    C    [2.97, 2.97]
    D    [2.98, 2.53]
    E    [2.97, 2.35]

  -- Final output (concat heads @ W_out)  [5, 4] --
    A    [3.0, 6.0, 3.0, 3.0]
    B    [2.91, 5.86, 2.92, 2.84]
    C    [2.95, 5.92, 2.97, 2.97]
    D    [2.71, 5.69, 2.98, 2.53]
    E    [2.51, 5.48, 2.97, 2.35]


**********************************************************************
**********************************************************************
  INFERENCE MODE — Autoregressive Decoding with KV Cache
**********************************************************************
**********************************************************************

######################################################################
  PREFILL: [A, B, C]
######################################################################

  Input shape: [1, 3, 4]  (B=1, S=3, d_model=4)
  Positions: [0, 1, 2]

  -- Input X  [3, 4] --
    A    [1.0, 0.0, 2.0, 1.0]
    B    [0.0, 1.0, 1.0, 0.0]
    C    [2.0, 1.0, 0.0, 1.0]

  --- Query Path ---

  -- q_content Head 0  [3, 2] --
    A    [3.0, 2.0]
    B    [1.0, 2.0]
    C    [2.0, 1.0]

  -- q_content Head 1  [3, 2] --
    A    [1.0, 2.0]
    B    [1.0, 0.0]
    C    [2.0, 3.0]

  -- q_rope (with position) Head 0  [3, 2] --
    A    [2.0, 1.0]
    B    [0.1, 1.1]
    C    [3.2, 2.2]

  -- q_rope (with position) Head 1  [3, 2] --
    A    [3.0, 2.0]
    B    [1.1, 2.1]
    C    [2.2, 1.2]

  --- KV Path: Compress ---

  -- c_kv_new  (X @ W_kv_down)  [3, 2] --
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]
  Compressed from 4 dims to 2 dims per token

  k_rope_new: [1, 1, 3, 2]

  --- KV Cache ---
  No past cache (first step)

  *** WHAT IS CACHED (per token): ***
    c_kv:   2 values   (compressed KV latent)
    k_rope: 2 values   (positional key)
    Total:  4 values per token
  *** Vs standard MHA: 8 values per token ***

  -- Cached c_kv (ALL 3 tokens)  [3, 2] --
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]

  --- Decompress c_kv -> K_content + V ---
  c_kv @ W_kv_up: [1, 3, 2] @ [2, 8] -> [1, 3, 8]
  (decompress 2 dims back to 8 per token)

  -- k_content (decompressed) Head 0  [3, 2] --
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]

  -- k_content (decompressed) Head 1  [3, 2] --
    A    [3.0, 3.0]
    B    [2.0, 1.0]
    C    [2.0, 2.0]

  -- v (decompressed) Head 0  [3, 2] --
    A    [3.0, 6.0]
    B    [1.0, 3.0]
    C    [2.0, 4.0]

  -- v (decompressed) Head 1  [3, 2] --
    A    [3.0, 3.0]
    B    [2.0, 1.0]
    C    [2.0, 2.0]

  --- Assemble q_full and k_full ---
  q_full = cat(q_content[2d], q_rope[2d]) -> [4d per head]
  k_full = cat(k_content[2d], k_rope[2d]) -> [4d per head]
  q_full shape: [1, 2, 3, 4]  (query: 3 tokens)
  k_full shape: [1, 2, 3, 4]  (keys: 3 tokens)

  --- Attention ---

  -- Head 0 Masked Scores --
    query v  key ->  ['A', 'B', 'C']
    A              [' 11.00', '  -inf', '  -inf']
    B              ['  6.25', '  3.21', '  -inf']
    C              [' 11.00', '  6.57', '  7.34']

  -- Head 1 Masked Scores --
    query v  key ->  ['A', 'B', 'C']
    A              [' 10.50', '  -inf', '  -inf']
    B              ['  5.75', '  3.31', '  -inf']
    C              [' 11.50', '  6.47', '  7.64']

  -- Head 0 Attention Weights --
    query v  key ->  ['A', 'B', 'C']
    A              ['  1.00', '  0.00', '  0.00']
    B              ['  0.95', '  0.05', '  0.00']
    C              ['  0.96', '  0.01', '  0.02']

  -- Head 1 Attention Weights --
    query v  key ->  ['A', 'B', 'C']
    A              ['  1.00', '  0.00', '  0.00']
    B              ['  0.92', '  0.08', '  0.00']
    C              ['  0.97', '  0.01', '  0.02']

  Output = weights @ V  (V is head_dim=2, not 4)

  -- Attention output Head 0  [3, 2] --
    A    [3.0, 6.0]
    B    [2.91, 5.86]
    C    [2.95, 5.92]

  -- Attention output Head 1  [3, 2] --
    A    [3.0, 3.0]
    B    [2.92, 2.84]
    C    [2.97, 2.97]

  -- Final output  [3, 4] --
    A    [3.0, 6.0, 3.0, 3.0]
    B    [2.91, 5.86, 2.92, 2.84]
    C    [2.95, 5.92, 2.97, 2.97]

######################################################################
  DECODE step 1: token 'D' (position 3)
######################################################################

  Input shape: [1, 1, 4]  (B=1, S=1, d_model=4)
  Positions: [3]

  -- Input X  [1, 4] --
    D    [1.0, 2.0, 1.0, 0.0]

  --- Query Path ---

  -- q_content Head 0  [1, 2] --
    D    [2.0, 3.0]

  -- q_content Head 1  [1, 2] --
    D    [2.0, 1.0]

  -- q_rope (with position) Head 0  [1, 2] --
    D    [1.3, 2.3]

  -- q_rope (with position) Head 1  [1, 2] --
    D    [2.3, 3.3]

  --- KV Path: Compress ---

  -- c_kv_new  (X @ W_kv_down)  [1, 2] --
    D    [2.0, 3.0]
  Compressed from 4 dims to 2 dims per token

  k_rope_new: [1, 1, 1, 2]

  --- KV Cache ---
  Past cache:
    c_kv shape:   [1, 3, 2]  (3 tokens: ['A', 'B', 'C'])
    k_rope shape: [1, 1, 3, 2]

  After concatenation:
    c_kv:   [1, 4, 2]  (tokens: ['A', 'B', 'C', 'D'])
    k_rope: [1, 1, 4, 2]

  *** WHAT IS CACHED (per token): ***
    c_kv:   2 values   (compressed KV latent)
    k_rope: 2 values   (positional key)
    Total:  4 values per token
  *** Vs standard MHA: 8 values per token ***

  -- Cached c_kv (ALL 4 tokens)  [4, 2] --
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]
    D    [2.0, 3.0]

  --- Decompress c_kv -> K_content + V ---
  c_kv @ W_kv_up: [1, 4, 2] @ [2, 8] -> [1, 4, 8]
  (decompress 2 dims back to 8 per token)

  -- k_content (decompressed) Head 0  [4, 2] --
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]
    D    [2.0, 3.0]

  -- k_content (decompressed) Head 1  [4, 2] --
    A    [3.0, 3.0]
    B    [2.0, 1.0]
    C    [2.0, 2.0]
    D    [3.0, 2.0]

  -- v (decompressed) Head 0  [4, 2] --
    A    [3.0, 6.0]
    B    [1.0, 3.0]
    C    [2.0, 4.0]
    D    [2.0, 5.0]

  -- v (decompressed) Head 1  [4, 2] --
    A    [3.0, 3.0]
    B    [2.0, 1.0]
    C    [2.0, 2.0]
    D    [3.0, 2.0]

  --- Assemble q_full and k_full ---
  q_full = cat(q_content[2d], q_rope[2d]) -> [4d per head]
  k_full = cat(k_content[2d], k_rope[2d]) -> [4d per head]
  q_full shape: [1, 2, 1, 4]  (query: 1 tokens)
  k_full shape: [1, 2, 4, 4]  (keys: 4 tokens)

  --- Attention ---

  -- Head 0 Masked Scores --
    query v  key ->  ['A', 'B', 'C', 'D']
    D              [' 12.25', '  6.63', '  8.31', ' 11.29']

  -- Head 1 Masked Scores --
    query v  key ->  ['A', 'B', 'C', 'D']
    D              [' 11.75', '  6.73', '  8.01', ' 11.59']

  -- Head 0 Attention Weights --
    query v  key ->  ['A', 'B', 'C', 'D']
    D              ['  0.71', '  0.00', '  0.01', '  0.27']

  -- Head 1 Attention Weights --
    query v  key ->  ['A', 'B', 'C', 'D']
    D              ['  0.53', '  0.00', '  0.01', '  0.45']

  Output = weights @ V  (V is head_dim=2, not 4)

  -- Attention output Head 0  [1, 2] --
    D    [2.71, 5.69]

  -- Attention output Head 1  [1, 2] --
    D    [2.98, 2.53]

  -- Final output  [1, 4] --
    D    [2.71, 5.69, 2.98, 2.53]

######################################################################
  DECODE step 2: token 'E' (position 4)
######################################################################

  Input shape: [1, 1, 4]  (B=1, S=1, d_model=4)
  Positions: [4]

  -- Input X  [1, 4] --
    E    [0.0, 0.0, 2.0, 1.0]

  --- Query Path ---

  -- q_content Head 0  [1, 2] --
    E    [2.0, 2.0]

  -- q_content Head 1  [1, 2] --
    E    [1.0, 1.0]

  -- q_rope (with position) Head 0  [1, 2] --
    E    [1.4, 1.4]

  -- q_rope (with position) Head 1  [1, 2] --
    E    [2.4, 2.4]

  --- KV Path: Compress ---

  -- c_kv_new  (X @ W_kv_down)  [1, 2] --
    E    [2.0, 3.0]
  Compressed from 4 dims to 2 dims per token

  k_rope_new: [1, 1, 1, 2]

  --- KV Cache ---
  Past cache:
    c_kv shape:   [1, 4, 2]  (4 tokens: ['A', 'B', 'C', 'D'])
    k_rope shape: [1, 1, 4, 2]

  After concatenation:
    c_kv:   [1, 5, 2]  (tokens: ['A', 'B', 'C', 'D', 'E'])
    k_rope: [1, 1, 5, 2]

  *** WHAT IS CACHED (per token): ***
    c_kv:   2 values   (compressed KV latent)
    k_rope: 2 values   (positional key)
    Total:  4 values per token
  *** Vs standard MHA: 8 values per token ***

  -- Cached c_kv (ALL 5 tokens)  [5, 2] --
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]
    D    [2.0, 3.0]
    E    [2.0, 3.0]

  --- Decompress c_kv -> K_content + V ---
  c_kv @ W_kv_up: [1, 5, 2] @ [2, 8] -> [1, 5, 8]
  (decompress 2 dims back to 8 per token)

  -- k_content (decompressed) Head 0  [5, 2] --
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]
    D    [2.0, 3.0]
    E    [2.0, 3.0]

  -- k_content (decompressed) Head 1  [5, 2] --
    A    [3.0, 3.0]
    B    [2.0, 1.0]
    C    [2.0, 2.0]
    D    [3.0, 2.0]
    E    [3.0, 2.0]

  -- v (decompressed) Head 0  [5, 2] --
    A    [3.0, 6.0]
    B    [1.0, 3.0]
    C    [2.0, 4.0]
    D    [2.0, 5.0]
    E    [2.0, 5.0]

  -- v (decompressed) Head 1  [5, 2] --
    A    [3.0, 3.0]
    B    [2.0, 1.0]
    C    [2.0, 2.0]
    D    [3.0, 2.0]
    E    [3.0, 2.0]

  --- Assemble q_full and k_full ---
  q_full = cat(q_content[2d], q_rope[2d]) -> [4d per head]
  k_full = cat(k_content[2d], k_rope[2d]) -> [4d per head]
  q_full shape: [1, 2, 1, 4]  (query: 1 tokens)
  k_full shape: [1, 2, 5, 4]  (keys: 5 tokens)

  --- Attention ---

  -- Head 0 Masked Scores --
    query v  key ->  ['A', 'B', 'C', 'D', 'E']
    E              ['  9.50', '  5.24', '  6.38', '  8.92', '  8.36']

  -- Head 1 Masked Scores --
    query v  key ->  ['A', 'B', 'C', 'D', 'E']
    E              ['  9.00', '  5.34', '  6.08', '  9.22', '  8.26']

  -- Head 0 Attention Weights --
    query v  key ->  ['A', 'B', 'C', 'D', 'E']
    E              ['  0.52', '  0.01', '  0.02', '  0.29', '  0.17']

  -- Head 1 Attention Weights --
    query v  key ->  ['A', 'B', 'C', 'D', 'E']
    E              ['  0.36', '  0.01', '  0.02', '  0.44', '  0.17']

  Output = weights @ V  (V is head_dim=2, not 4)

  -- Attention output Head 0  [1, 2] --
    E    [2.51, 5.48]

  -- Attention output Head 1  [1, 2] --
    E    [2.97, 2.35]

  -- Final output  [1, 4] --
    E    [2.51, 5.48, 2.97, 2.35]


======================================================================
  SUMMARY: MLA vs Standard Attention KV Cache
======================================================================

  What MLA caches per token:
    c_kv   (2 values) — compressed latent, contains ALL K/V info
    k_rope (2 values) — positional key (shared across heads)
    TOTAL: 4 values

  What standard MHA would cache per token:
    K (2 heads x 2 dim) = 4 values
    V (2 heads x 2 dim) = 4 values
    TOTAL: 8 values

  Savings in this toy: 8/4 = 2.0x

  Trade-off: Every decode step must decompress c_kv -> K, V via W_kv_up
  for ALL cached tokens (not just the new one). This costs compute but
  saves memory — a good trade when you're memory-bound (long sequences).

  Why decoupled RoPE?
  -------------------
  If we applied RoPE INSIDE the latent (before compressing), the position
  info would be baked into c_kv and we couldn't cache a position-free
  latent. If we applied it AFTER decompressing, we'd need to decompress
  first to apply it, which defeats the purpose.

  Solution: a separate lightweight projection (k_rope_proj) that carries
  ONLY the positional signal. It's concatenated with the content key at
  attention time:
    score = [q_content | q_rope] @ [k_content | k_rope]^T
          = q_content @ k_content^T  +  q_rope @ k_rope^T
            ^^^^^^^^^^^^^^^^^^^^        ^^^^^^^^^^^^^^^^^^^^
            content similarity          positional similarity